In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import zipfile

zip_path = "/content/drive/MyDrive/Colab Notebooks/DeepRet/DL.zip"
extract_path = "/content/dataset"

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset extracted successfully!")

Dataset extracted successfully!


In [ ]:
import os

print(os.listdir("/content/dataset"))

['ODIR-5K']


In [ ]:
import pandas as pd

train_df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/DeepRet/train.csv")
val_df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/DeepRet/validation.csv")

print("Training Images :", len(train_df))
print("Validation Images :", len(val_df))

train_df.head()

Training Images : 10227
Validation Images : 2557


,filename,N,D,G,C,A,H,M,O
0,4080_left.jpg,0,1,0,0,0,0,0,0
1,64_left.jpg,0,1,0,0,0,0,0,1
2,2873_right.jpg,1,0,0,0,0,0,0,0
3,2419_right.jpg,1,0,0,0,0,0,0,0
4,862_left.jpg,0,0,0,0,0,0,0,1


In [ ]:
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import (
    Input,
    GlobalAveragePooling2D,
    Dense,
    Dropout
)
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.metrics import BinaryAccuracy, AUC

In [ ]:
# Build ResNet50 Model

input_tensor = Input(shape=(224, 224, 3))

base_model = ResNet50(
    weights="imagenet",
    include_top=False,
    input_tensor=input_tensor
)

# Freeze the pretrained layers
base_model.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dropout(0.5)(x)
x = Dense(256, activation="relu")(x)
x = Dropout(0.3)(x)

output = Dense(8, activation="sigmoid")(x)

model = Model(inputs=base_model.input, outputs=output)

model.summary()

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_pad           │ (None, 230, 230,  │          0 │ input_layer[0][0] │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_conv (Conv2D) │ (None, 112, 112,  │      9,472 │ conv1_pad[0][0]   │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_bn            │ (None, 112, 112,  │        256 │ conv1_conv[0][0]  │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_relu          │ (None, 112, 112,  │          0 │ conv1_bn[0][0]    │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pad           │ (None, 114, 114,  │          0 │ conv1_relu[0][0]  │
│ (ZeroPadding2D)     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pool          │ (None, 56, 56,    │          0 │ pool1_pad[0][0]   │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_conv │ (None, 56, 56,    │      4,160 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_bn   │ (None, 56, 56,    │        256 │ conv2_block1_1_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_relu │ (None, 56, 56,    │          0 │ conv2_block1_1_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_conv │ (None, 56, 56,    │     36,928 │ conv2_block1_1_r… │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_bn   │ (None, 56, 56,    │        256 │ conv2_block1_2_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_relu │ (None, 56, 56,    │          0 │ conv2_block1_2_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_conv │ (None, 56, 56,    │     16,640 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_conv │ (None, 56, 56,    │     16,640 │ conv2_block1_2_r… │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_bn   │ (None, 56, 56,    │      1,024 │ conv2_block1_0_c… │
│ (BatchNormalizatio… │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_bn   │ (None, 56, 56,    │      1,024 │ conv2_block1_3_c

 Total params: 24,114,312 (91.99 MB)

 Trainable params: 526,600 (2.01 MB)

 Non-trainable params: 23,587,712 (89.98 MB)

In [ ]:
model.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss="binary_crossentropy",
    metrics=[
        BinaryAccuracy(name="bin_acc"),
        AUC(name="roc_auc", multi_label=True)
    ]
)

print("ResNet50 model compiled successfully!")

ResNet50 model compiled successfully!


In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

checkpoint_path = "/content/drive/MyDrive/Colab Notebooks/DeepRet/resnet50_best.keras"

callbacks = [
    EarlyStopping(
        monitor="val_loss",
        patience=8,
        restore_best_weights=True,
        verbose=1
    ),

    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.2,
        patience=4,
        min_lr=1e-7,
        verbose=1
    ),

    ModelCheckpoint(
        checkpoint_path,
        monitor="val_loss",
        save_best_only=True,
        verbose=1
    )
]

print("Callbacks created successfully!")

Callbacks created successfully!


In [ ]:
import tensorflow as tf

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

def load_image(filename, labels):
    image = tf.io.read_file("/content/dataset/ODIR-5K/" + filename)
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, IMG_SIZE)
    image = image / 255.0
    return image, labels

# Labels
train_labels = train_df.iloc[:, 1:].values.astype("float32")
val_labels = val_df.iloc[:, 1:].values.astype("float32")

# Datasets
train_dataset = tf.data.Dataset.from_tensor_slices(
    (train_df["filename"].values, train_labels)
)

val_dataset = tf.data.Dataset.from_tensor_slices(
    (val_df["filename"].values, val_labels)
)

train_dataset = (
    train_dataset
    .map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
    .shuffle(1000)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

val_dataset = (
    val_dataset
    .map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

print("Datasets created successfully!")

Datasets created successfully!


In [ ]:
import os

print("Checking dataset folder...")

for root, dirs, files in os.walk("/content/dataset"):
    if len(files) > 0:
        print("\nFolder:", root)
        print("First 10 files:")
        print(files[:10])
        break

Checking dataset folder...

Folder: /content/dataset/ODIR-5K
First 10 files:
['data.xlsx', 'full_df.csv']


In [ ]:
import os

for root, dirs, files in os.walk("/content/dataset/ODIR-5K"):
    jpgs = [f for f in files if f.endswith(".jpg")]
    if jpgs:
        print("Image folder:", root)
        print("First 10 images:", jpgs[:10])

Image folder: /content/dataset/ODIR-5K/training
First 10 images: ['999_left.jpg', '3293_left.jpg', '4202_right.jpg', '2451_right.jpg', '2895_right.jpg', '829_right.jpg', '1299_left.jpg', '271_left.jpg', '513_left.jpg', '2801_right.jpg']
Image folder: /content/dataset/ODIR-5K/testing
First 10 images: ['1680_left.jpg', '4693_right.jpg', '3571_right.jpg', '1324_right.jpg', '4720_right.jpg', '4696_left.jpg', '4766_left.jpg', '3468_left.jpg', '3528_right.jpg', '1298_left.jpg']


In [ ]:
train_exists = train_df["filename"].apply(
    lambda x: os.path.exists(f"/content/dataset/ODIR-5K/training/{x}")
).sum()

test_exists = train_df["filename"].apply(
    lambda x: os.path.exists(f"/content/dataset/ODIR-5K/testing/{x}")
).sum()

print("Training folder matches:", train_exists)
print("Testing folder matches :", test_exists)

Training folder matches: 10227
Testing folder matches : 0


In [ ]:
import tensorflow as tf

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

# Labels
train_labels = train_df.iloc[:, 1:].values.astype("float32")
val_labels = val_df.iloc[:, 1:].values.astype("float32")

# Image loading function
def load_image(filename, labels):
    image_path = tf.strings.join(
        ["/content/dataset/ODIR-5K/training/", filename]
    )

    image = tf.io.read_file(image_path)
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, IMG_SIZE)
    image = tf.cast(image, tf.float32) / 255.0

    return image, labels

# Create datasets
train_dataset = tf.data.Dataset.from_tensor_slices(
    (train_df["filename"].values, train_labels)
)

val_dataset = tf.data.Dataset.from_tensor_slices(
    (val_df["filename"].values, val_labels)
)

train_dataset = (
    train_dataset
    .map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
    .shuffle(1000)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

val_dataset = (
    val_dataset
    .map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

print("Datasets created successfully!")

# Train model
history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=10,
    callbacks=callbacks
)